# Feature engineering v2 (adds history-based features)
Same pipeline as `new_feature_engineering.ipynb`, plus one extra cell with product, category, user and user-product history features.

**Safe to run:** it only READS `../data/reconstructed_events.csv` and writes NEW files `../data/cart_labels_v2.csv` and `../data/final_model_ready_cart_events_v2.csv`. The old CSV files are not touched.

In [1]:
import numpy as np
import pandas as pd

In [2]:
dtypes = {
    "user_id": "int64",
    "event_type": "category",
    "product_id": "int64",
    "category_id": "int64",
    "brand": "category",
    "price": "float32",
    "brand_imputed": "bool",
    "brand_fallback": "bool",
    "session_id": "int32",
}

df = pd.read_csv("../data/reconstructed_events.csv", dtype=dtypes)

In [3]:
# Saved times look like "2019-10-01 00:00:00+00:00"
df["event_time"] = pd.to_datetime(df["event_time"], format="%Y-%m-%d %H:%M:%S%z", utc=True)

# Rebuild new_session: True on the first event of every session
df["new_session"] = df["session_id"].ne(df["session_id"].shift(1))

# Checks (expected: 11,835,068 rows, 1,011,684 users, 1,937,785 sessions)
print("Shape:", df.shape)
print("Unique users:", df["user_id"].nunique())
print("Sessions:", df["session_id"].nunique(), "| new_session count:", int(df["new_session"].sum()))
print("Session ids contiguous:", df["session_id"].iloc[-1] == df["session_id"].nunique())
print("Sorted by user and time:",
      bool(((df["user_id"].diff().fillna(0) > 0) |
            (df["event_time"].diff().fillna(pd.Timedelta(0)) >= pd.Timedelta(0))).all()))
print(df.dtypes)
print(df.head())

Shape: (11835068, 11)
Unique users: 1011684
Sessions: 1937785 | new_session count: 1937785
Session ids contiguous: True
Sorted by user and time: True
user_id                         int64
event_time        datetime64[us, UTC]
event_type                   category
product_id                      int64
category_id                     int64
brand                        category
price                         float32
brand_imputed                    bool
brand_fallback                   bool
session_id                      int32
new_session                      bool
dtype: object
   user_id                event_time event_type  product_id  \
0  1120748 2019-11-04 15:28:57+00:00       view        5240   
1  1180452 2019-12-28 14:32:56+00:00       view     5881337   
2  1458813 2019-11-19 09:56:09+00:00       view     5711145   
3  1458813 2019-11-19 11:14:48+00:00       view     5814066   
4  1458813 2019-11-20 11:58:06+00:00       view     5800358   

           category_id    brand  price 

In [4]:
WINDOW_DAYS = 7
window = pd.Timedelta(days=WINDOW_DAYS)
data_end = df["event_time"].max()

# Cart events, keeping the row position of df as row_id
cart_ev = (
    df.loc[df["event_type"] == "cart",
           ["user_id", "product_id", "category_id", "price", "event_time", "session_id"]]
    .reset_index()
    .rename(columns={"index": "row_id"})
    .sort_values("event_time")
)

# Purchase and remove events of the same user and product
pur_ev = (
    df.loc[df["event_type"] == "purchase", ["user_id", "product_id", "event_time"]]
    .rename(columns={"event_time": "purchase_time"})
    .sort_values("purchase_time")
)
rem_ev = (
    df.loc[df["event_type"] == "remove_from_cart", ["user_id", "product_id", "event_time"]]
    .rename(columns={"event_time": "remove_time"})
    .sort_values("remove_time")
)

# First later purchase and first later remove for every cart event
lbl = pd.merge_asof(cart_ev, pur_ev, left_on="event_time", right_on="purchase_time",
                    by=["user_id", "product_id"], direction="forward")
lbl = pd.merge_asof(lbl, rem_ev, left_on="event_time", right_on="remove_time",
                    by=["user_id", "product_id"], direction="forward")

# Keep only events that happen inside the window
p = lbl["purchase_time"].where(lbl["purchase_time"] - lbl["event_time"] <= window)
r = lbl["remove_time"].where(lbl["remove_time"] - lbl["event_time"] <= window)

# Whichever happens first decides the raw outcome
is_cleared = r.notna() & (p.isna() | (r < p))
is_purchased = p.notna() & ~is_cleared
outcome_raw = np.select([is_purchased, is_cleared], ["purchased", "cleared"], default="abandoned")
lbl["outcome_raw"] = outcome_raw

# Cleared carts that were still bought inside the window become "purchased"
rebought = (outcome_raw == "cleared") & p.notna().to_numpy()
lbl["outcome"] = np.where(rebought, "purchased", outcome_raw)
lbl["removed_then_rebought"] = rebought

# Quick removes: removed within 1 minute of the cart event
remove_lag_min = (lbl["remove_time"] - lbl["event_time"]).dt.total_seconds() / 60
lbl["quick_remove"] = (outcome_raw == "cleared") & (remove_lag_min <= 1).to_numpy()

# Abandoned carts near the end of the data did not get a full window
lbl["censored"] = (lbl["outcome"] == "abandoned") & (lbl["event_time"] + window > data_end)

# Binary target: 1 = not bought (abandoned or cleared), 0 = purchased
lbl["is_abandoned"] = lbl["outcome"].isin(["abandoned", "cleared"]).astype("int8")

# Keep label columns only (lookahead columns are dropped to avoid leakage)
labels = lbl[[
    "row_id", "session_id", "user_id", "product_id", "category_id", "price",
    "event_time", "outcome", "is_abandoned", "censored",
    "quick_remove", "removed_then_rebought",
]].copy()
del cart_ev, pur_ev, rem_ev, lbl, p, r

# Checks (expected: abandoned 1,396,029 | cleared 1,136,913 | purchased 888,791)
print("Cart events:", len(labels))
print(labels["outcome"].value_counts())
print("\nCensored:", int(labels["censored"].sum()), "(expected 62,464)")
print("Quick removes:", int(labels["quick_remove"].sum()), "(expected 154,085)")
print("Cleared -> rebought:", int(labels["removed_then_rebought"].sum()), "(expected 151,400)")
print("is_abandoned share (%):")
print((labels.loc[~labels["censored"], "is_abandoned"].value_counts(normalize=True) * 100).round(2))

# Save labels so they never need to be rebuilt
labels.to_csv("../data/cart_labels_v2.csv", index=False)
print("\nSaved: ../data/cart_labels_v2.csv")

Cart events: 3421733
outcome
abandoned    1396029
cleared      1136913
purchased     888791
Name: count, dtype: int64

Censored: 62464 (expected 62,464)
Quick removes: 154085 (expected 154,085)
Cleared -> rebought: 151400 (expected 151,400)
is_abandoned share (%):
is_abandoned
1    73.54
0    26.46
Name: proportion, dtype: float64

Saved: ../data/cart_labels_v2.csv


In [5]:
sid = df["session_id"].to_numpy()
starts = np.flatnonzero(df["new_session"].to_numpy())   # first row of every session (session_id - 1 indexes it)

# Event type as integer codes (much lighter than strings)
codes = df["event_type"].cat.codes.to_numpy()
cats = list(df["event_type"].cat.categories)

# Running count of one event type inside each session (includes the current row)
def cum_in_session(event_name):
    flag = (codes == cats.index(event_name)).astype(np.int32)
    c = np.cumsum(flag, dtype=np.int32)
    before_session = c[starts] - flag[starts]      # count before each session begins
    return c - before_session[sid - 1]

cum_view = cum_in_session("view")
cum_cart = cum_in_session("cart")
cum_remove = cum_in_session("remove_from_cart")
cum_purchase = cum_in_session("purchase")

# Event time in seconds since epoch
t_sec = (df["event_time"] - pd.Timestamp("1970-01-01", tz="UTC")).dt.total_seconds().to_numpy()

# Features for cart events only (row_id is the row position in df)
rows = labels["row_id"].to_numpy()
sess_start_sec = t_sec[starts[sid[rows] - 1]]

feat = pd.DataFrame({
    "row_id": rows,
    "views_before_cart": cum_view[rows],                 # views so far in this session
    "carts_before": cum_cart[rows] - 1,                  # earlier cart adds (current one excluded)
    "removes_before": cum_remove[rows],                  # removes so far in this session
    "purchases_before_in_session": cum_purchase[rows],   # purchases so far in this session
    "session_age_min": ((t_sec[rows] - sess_start_sec) / 60).astype("float32"),
})
feat["is_first_cart_in_session"] = (feat["carts_before"] == 0).astype("int8")
feat["no_view_before_cart"] = (feat["views_before_cart"] == 0).astype("int8")

# Time features from the cart event time (UTC)
ct = labels["event_time"].reset_index(drop=True)
feat["cart_hour"] = ct.dt.hour.to_numpy().astype("int8")
feat["cart_day_of_week"] = ct.dt.dayofweek.to_numpy().astype("int8")
feat["cart_is_weekend"] = (feat["cart_day_of_week"] >= 5).astype("int8")

# Checks
print("Feature rows:", feat.shape)
print("Nulls:", int(feat.isnull().sum().sum()))
print("Rows match labels:", len(feat) == len(labels))
print("Negative counts:", int((feat[["views_before_cart", "carts_before", "removes_before"]] < 0).sum().sum()))
print(feat.drop(columns="row_id").describe().T[["mean", "50%", "max"]])

# First look at how the target changes with simple features
chk = feat.assign(is_abandoned=labels["is_abandoned"].to_numpy())
print("\nAbandoned share (%) by views before cart:")
print((chk.groupby(pd.cut(chk["views_before_cart"], [-1, 0, 2, 5, 10, 100000]), observed=True)["is_abandoned"].mean() * 100).round(1))
print("\nAbandoned share (%) by first cart in session:")
print((chk.groupby("is_first_cart_in_session")["is_abandoned"].mean() * 100).round(1))
print("\nAbandoned share (%) by removes before this cart:")
print((chk.groupby(pd.cut(chk["removes_before"], [-1, 0, 2, 5, 100000]), observed=True)["is_abandoned"].mean() * 100).round(1))

Feature rows: (3421733, 11)
Nulls: 0
Rows match labels: True
Negative counts: 0
                                  mean        50%          max
views_before_cart             4.982993   1.000000  1047.000000
carts_before                 12.859325   5.000000  1278.000000
removes_before                4.336389   0.000000  1525.000000
purchases_before_in_session   0.131992   0.000000   279.000000
session_age_min              16.394602   7.716667   650.016663
is_first_cart_in_session      0.154853   0.000000     1.000000
no_view_before_cart           0.295571   0.000000     1.000000
cart_hour                    12.983899  13.000000    23.000000
cart_day_of_week              2.930755   3.000000     6.000000
cart_is_weekend               0.262958   0.000000     1.000000

Abandoned share (%) by views before cart:
views_before_cart
(-1, 0]         76.3
(0, 2]          77.3
(2, 5]          71.6
(5, 10]         69.9
(10, 100000]    66.7
Name: is_abandoned, dtype: float64

Abandoned share (%) by fi

In [6]:
price = df["price"].to_numpy(dtype="float64")
codes = df["event_type"].cat.codes.to_numpy()
cats = list(df["event_type"].cat.categories)
rows = labels["row_id"].to_numpy()

# 1. Net cart value inside the session (cart adds minus removes) up to the current cart event
signed = np.zeros(len(df))
is_cart = codes == cats.index("cart")
is_remove = codes == cats.index("remove_from_cart")
signed[is_cart] = price[is_cart]
signed[is_remove] = -price[is_remove]
c = np.cumsum(signed)
before_session = c[starts] - signed[starts]
net_value = c - before_session[sid - 1]
feat["net_cart_value"] = np.clip(net_value[rows], 0, None).astype("float32")   # negative = cart came from an earlier session
del signed, c, before_session, net_value

# 2. Item price versus its category average (catalog-level statistic, no outcome used)
cat_avg = df.groupby("category_id")["price"].mean()
feat["price"] = labels["price"].to_numpy()
feat["category_avg_price"] = labels["category_id"].map(cat_avg).to_numpy().astype("float32")
feat["price_vs_category_avg"] = (feat["price"] - feat["category_avg_price"]).astype("float32")
feat["price_ratio_to_category"] = (feat["price"] / feat["category_avg_price"]).astype("float32")

# 3. Price change between the user's last earlier view of the same product and the cart event
views = (
    df.loc[df["event_type"] == "view", ["user_id", "product_id", "event_time", "price"]]
    .rename(columns={"event_time": "view_time", "price": "last_view_price"})
    .sort_values("view_time")
)
carts = pd.DataFrame({
    "row_id": rows,
    "user_id": labels["user_id"].to_numpy(),
    "product_id": labels["product_id"].to_numpy(),
    "event_time": labels["event_time"].to_numpy(),
}).sort_values("event_time")

m = pd.merge_asof(
    carts, views,
    left_on="event_time", right_on="view_time",
    by=["user_id", "product_id"],
    direction="backward", allow_exact_matches=False,
)
m = m.set_index("row_id").reindex(rows)

feat["has_prior_view_of_product"] = m["view_time"].notna().to_numpy().astype("int8")
feat["price_change_since_view"] = (feat["price"] - m["last_view_price"].to_numpy()).astype("float32")
feat["hours_since_last_view"] = ((m["event_time"] - m["view_time"]).dt.total_seconds() / 3600).to_numpy().astype("float32")

# Fill missing values with neutral placeholders (the flag column keeps the information)
feat["price_change_since_view"] = feat["price_change_since_view"].fillna(0)
feat["hours_since_last_view"] = feat["hours_since_last_view"].fillna(-1)
del views, carts, m

# Checks
new_cols = ["net_cart_value", "category_avg_price", "price_vs_category_avg",
            "price_ratio_to_category", "has_prior_view_of_product",
            "price_change_since_view", "hours_since_last_view"]
print("Feature shape:", feat.shape)
print("Nulls in new columns:", int(feat[new_cols].isnull().sum().sum()))
print(feat[new_cols].describe().T[["mean", "50%", "max"]])
print("\nCarts where price changed since last view (%):",
      round(((feat["price_change_since_view"] != 0) & (feat["has_prior_view_of_product"] == 1)).mean() * 100, 2))

# How the target moves with the new features
chk = feat.assign(is_abandoned=labels["is_abandoned"].to_numpy())
print("\nAbandoned share (%) by price ratio to category:")
print((chk.groupby(pd.qcut(chk["price_ratio_to_category"], 5, duplicates="drop"), observed=True)["is_abandoned"].mean() * 100).round(1))
print("\nAbandoned share (%) by prior view of the same product:")
print((chk.groupby("has_prior_view_of_product")["is_abandoned"].mean() * 100).round(1))
print("\nAbandoned share (%) by price change since view:")
sign = np.sign(chk["price_change_since_view"]).where(chk["has_prior_view_of_product"] == 1)
print((chk.groupby(sign)["is_abandoned"].mean() * 100).round(1))
print("\nAbandoned share (%) by net cart value:")
print((chk.groupby(pd.qcut(chk["net_cart_value"], 5, duplicates="drop"), observed=True)["is_abandoned"].mean() * 100).round(1))

Feature shape: (3421733, 19)
Nulls in new columns: 0
                                mean        50%          max
net_cart_value             45.845112  15.080000  9096.530273
category_avg_price          5.869359   4.490281   164.884552
price_vs_category_avg      -0.644753  -0.332390   281.329742
price_ratio_to_category     0.906145   0.878241    44.707108
has_prior_view_of_product   0.263225   0.000000     1.000000
price_change_since_view    -0.000975   0.000000    70.949997
hours_since_last_view       6.990952  -1.000000  2179.544678

Carts where price changed since last view (%): 0.43

Abandoned share (%) by price ratio to category:
price_ratio_to_category
(0.00358, 0.49]    67.8
(0.49, 0.759]      72.4
(0.759, 0.967]     76.3
(0.967, 1.129]     78.3
(1.129, 44.707]    75.3
Name: is_abandoned, dtype: float64

Abandoned share (%) by prior view of the same product:
has_prior_view_of_product
0    75.3
1    70.4
Name: is_abandoned, dtype: float64

Abandoned share (%) by price change sinc

In [7]:
n = len(df)
sid = df["session_id"].to_numpy()
starts = np.flatnonzero(df["new_session"].to_numpy())
uid = df["user_id"].to_numpy()
codes = df["event_type"].cat.codes.to_numpy()
cats = list(df["event_type"].cat.categories)
rows = labels["row_id"].to_numpy()
t_sec = (df["event_time"] - pd.Timestamp("1970-01-01", tz="UTC")).dt.total_seconds().to_numpy()

# Group boundaries for users and sessions
user_start_flag = np.r_[True, uid[1:] != uid[:-1]]
ustarts = np.flatnonzero(user_start_flag)
uidx = np.cumsum(user_start_flag) - 1

# Running count of a 0/1 flag inside a group (includes the current row)
def cum_in_group(flag, group_starts, group_index):
    c = np.cumsum(flag, dtype=np.int32)
    before = c[group_starts] - flag[group_starts]
    return c - before[group_index]

sidx = sid - 1   # session index for every row

# 1. User history (all earlier sessions + current session before the cart)
is_purchase = (codes == cats.index("purchase")).astype(np.int32)
is_cart = (codes == cats.index("cart")).astype(np.int32)

purch_user = cum_in_group(is_purchase, ustarts, uidx)
cart_user = cum_in_group(is_cart, ustarts, uidx)
cart_sess = cum_in_group(is_cart, starts, sidx)

feat["previous_purchase_count"] = purch_user[rows]                      # purchases before this cart (any session)
feat["has_previous_purchase"] = (feat["previous_purchase_count"] > 0).astype("int8")
feat["previous_cart_events"] = (cart_user[rows] - cart_sess[rows])      # cart events in earlier sessions

# Session number of the user and time since the user's first event and previous session
session_rank = cum_in_group(user_start_flag.astype(np.int32) * 0 + (np.r_[True, sid[1:] != sid[:-1]]).astype(np.int32), ustarts, uidx)
feat["previous_sessions"] = (session_rank[rows] - 1)
feat["days_since_first_event"] = ((t_sec[rows] - t_sec[ustarts][uidx[rows]]) / 86400).astype("float32")

prev_gap_h = np.full(n, -1.0, dtype="float32")
sess_first = starts
has_prev = sess_first > 0
same_user_prev = np.zeros(len(sess_first), dtype=bool)
same_user_prev[has_prev] = uid[sess_first[has_prev]] == uid[sess_first[has_prev] - 1]
gap_h = np.full(len(sess_first), -1.0, dtype="float32")
gap_h[same_user_prev] = (t_sec[sess_first[same_user_prev]] - t_sec[sess_first[same_user_prev] - 1]) / 3600
feat["hours_since_prev_session"] = gap_h[sidx[rows]]                    # -1 = first session of the user

# 2. Category diversity in the session up to the cart (first time a category appears in a session)
first_cat = (~df.duplicated(["session_id", "category_id"])).to_numpy().astype(np.int32)
feat["category_diversity_before_cart"] = cum_in_group(first_cat, starts, sidx)[rows]

# 3. Brand switches in the session, real brands only (imputed brands are ignored)
is_view = (codes == cats.index("view"))
real_brand = ~df["brand_imputed"].to_numpy()
sub = np.flatnonzero(real_brand & (is_view | is_cart.astype(bool)))
bcodes = df["brand"].cat.codes.to_numpy()
s_sub, b_sub = sid[sub], bcodes[sub]
switch = np.zeros(len(sub), dtype=bool)
switch[1:] = (s_sub[1:] == s_sub[:-1]) & (b_sub[1:] != b_sub[:-1])
switch_flag = np.zeros(n, dtype=np.int32)
switch_flag[sub] = switch
feat["brand_switch_count_before_cart"] = cum_in_group(switch_flag, starts, sidx)[rows]
feat["cart_brand_is_real"] = real_brand[rows].astype("int8")

# Checks
new_cols = ["previous_purchase_count", "has_previous_purchase", "previous_cart_events",
            "previous_sessions", "days_since_first_event", "hours_since_prev_session",
            "category_diversity_before_cart", "brand_switch_count_before_cart", "cart_brand_is_real"]
print("Feature shape:", feat.shape)
print("Nulls in new columns:", int(feat[new_cols].isnull().sum().sum()))
print("Negative values:", int((feat[["previous_purchase_count", "previous_cart_events", "previous_sessions"]] < 0).sum().sum()))
print(feat[new_cols].describe().T[["mean", "50%", "max"]])

# How the target moves with the new features
chk = feat.assign(is_abandoned=labels["is_abandoned"].to_numpy())
print("\nAbandoned share (%) by previous purchases:")
print((chk.groupby(pd.cut(chk["previous_purchase_count"], [-1, 0, 2, 10, 100000]), observed=True)["is_abandoned"].mean() * 100).round(1))
print("\nAbandoned share (%) by previous sessions:")
print((chk.groupby(pd.cut(chk["previous_sessions"], [-1, 0, 2, 10, 100000]), observed=True)["is_abandoned"].mean() * 100).round(1))
print("\nAbandoned share (%) by category diversity:")
print((chk.groupby(pd.cut(chk["category_diversity_before_cart"], [0, 1, 2, 4, 100000]), observed=True)["is_abandoned"].mean() * 100).round(1))
print("\nAbandoned share (%) by brand switches (real brands):")
print((chk.groupby(pd.cut(chk["brand_switch_count_before_cart"], [-1, 0, 2, 5, 100000]), observed=True)["is_abandoned"].mean() * 100).round(1))

Feature shape: (3421733, 28)
Nulls in new columns: 0
Negative values: 0
                                     mean       50%          max
previous_purchase_count          5.193982  0.000000   398.000000
has_previous_purchase            0.219619  0.000000     1.000000
previous_cart_events            34.514126  2.000000  2966.000000
previous_sessions                5.323596  1.000000   363.000000
days_since_first_event          14.645147  1.888148    91.663330
hours_since_prev_session        82.426582  2.170833  2169.494385
category_diversity_before_cart   4.931719  3.000000   108.000000
brand_switch_count_before_cart   3.106137  1.000000   311.000000
cart_brand_is_real               0.575455  1.000000     1.000000

Abandoned share (%) by previous purchases:
previous_purchase_count
(-1, 0]         76.1
(0, 2]          62.1
(2, 10]         66.7
(10, 100000]    66.9
Name: is_abandoned, dtype: float64

Abandoned share (%) by previous sessions:
previous_sessions
(-1, 0]         77.3
(0, 2]   

In [8]:
# ---------- New history-based features (no outcome labels used, no future information) ----------
n = len(df)
sid = df["session_id"].to_numpy()
sidx = sid - 1
starts = np.flatnonzero(df["new_session"].to_numpy())
uid = df["user_id"].to_numpy().astype(np.int64)
pid = df["product_id"].to_numpy().astype(np.int64)
cid = df["category_id"].to_numpy().astype(np.int64)
codes = df["event_type"].cat.codes.to_numpy()
cats = list(df["event_type"].cat.categories)
rows = labels["row_id"].to_numpy()
t_sec = (df["event_time"] - pd.Timestamp("1970-01-01", tz="UTC")).dt.total_seconds().to_numpy()

EVENTS = ["view", "cart", "remove_from_cart", "purchase"]

def prior_counts(entity_key, rows):
    """For every row in `rows`, count the EARLIER events of each type that share the same
    entity (product, category, user, user+product). Order is global time, then row order."""
    order = np.lexsort((np.arange(n), t_sec, entity_key))   # entity -> time -> row
    ent = entity_key[order]
    first = np.r_[True, ent[1:] != ent[:-1]]
    gstarts = np.flatnonzero(first)
    gidx = np.cumsum(first) - 1
    pos = np.empty(n, dtype=np.int64)
    pos[order] = np.arange(n)                                # sorted position of each original row
    sorted_codes = codes[order]
    out = {}
    for ev in EVENTS:
        flag = (sorted_codes == cats.index(ev)).astype(np.int32)
        c = np.cumsum(flag, dtype=np.int64)
        within = c - (c[gstarts] - flag[gstarts])[gidx]      # inclusive count inside the entity
        before = within - flag                               # strictly before the current row
        out[ev] = before[pos[rows]].astype(np.int32)
    return out

assert pid.max() < 10_000_000, "product_id too large for the user+product key"
keys = {
    "prod": pid,
    "cat": cid,
    "user": uid,
    "userprod": uid * 10_000_000 + pid,
}

new = {}
for name, key in keys.items():
    pc = prior_counts(key, rows)
    carts = pc["cart"].astype("float32")
    new[f"{name}_prior_views"] = pc["view"]
    new[f"{name}_prior_carts"] = pc["cart"]
    new[f"{name}_prior_removes"] = pc["remove_from_cart"]
    new[f"{name}_prior_purchases"] = pc["purchase"]
    # Smoothed rates (the +5 keeps tiny counts from producing extreme ratios)
    new[f"{name}_prior_buy_rate"] = (pc["purchase"] / (carts + 5)).astype("float32")
    new[f"{name}_prior_remove_rate"] = (pc["remove_from_cart"] / (carts + 5)).astype("float32")
    new[f"{name}_prior_view_to_cart"] = (carts / (pc["view"] + 5)).astype("float32")
    del pc

# Session timing and cart size
prev_t = np.r_[t_sec[0], t_sec[:-1]]
same_sess = np.r_[False, sid[1:] == sid[:-1]]
gap = np.where(same_sess, t_sec - prev_t, -1.0)
new["secs_since_prev_event"] = gap[rows].astype("float32")
new["events_in_session_before"] = (np.arange(n) - starts[sidx])[rows].astype("int32")
new["cart_items_net"] = np.clip(feat["carts_before"].to_numpy() - feat["removes_before"].to_numpy(), 0, None)

# Price compared with the usual price of the same product (a drop can signal a discount)
prod_median = df.groupby("product_id")["price"].transform("median").to_numpy()
new["price_ratio_to_product_median"] = (feat["price"].to_numpy() / prod_median[rows]).astype("float32")

# Calendar position inside the data (captures sale periods such as Black Friday)
ct = labels["event_time"].reset_index(drop=True)
new["cart_day_of_year"] = ct.dt.dayofyear.to_numpy().astype("int16")

for col, values in new.items():
    feat[col] = values
del new, keys, prod_median, prev_t, same_sess, gap

print("Feature shape:", feat.shape)
print("Nulls:", int(feat.isnull().sum().sum()))


Feature shape: (3421733, 61)
Nulls: 0


In [9]:
# 1. Heavy sessions and users (possible bots or bulk carting)
cart_per_session = labels.groupby("session_id").size()
print("Cart events per session (percentiles):")
print(cart_per_session.describe(percentiles=[0.5, 0.9, 0.99, 0.999]))

heavy_sessions = cart_per_session[cart_per_session > 100].index
is_heavy = labels["session_id"].isin(heavy_sessions)
print("\nSessions with more than 100 cart events:", len(heavy_sessions))
print("Share of all cart events in those sessions (%):", round(is_heavy.mean() * 100, 2))
print("Abandoned share, heavy vs normal sessions (%):")
print((labels.groupby(is_heavy)["is_abandoned"].mean() * 100).round(1))

cart_per_user = labels.groupby("user_id").size()
heavy_users = cart_per_user[cart_per_user > 500].index
is_heavy_user = labels["user_id"].isin(heavy_users)
print("\nUsers with more than 500 cart events:", len(heavy_users))
print("Share of all cart events from those users (%):", round(is_heavy_user.mean() * 100, 2))
print("Abandoned share, heavy vs normal users (%):")
print((labels.groupby(is_heavy_user)["is_abandoned"].mean() * 100).round(1))

# 2. Correlation and skew on a random sample (keeps RAM low)
num_cols = [c for c in feat.columns if c != "row_id"]
idx = np.random.default_rng(42).choice(len(feat), 500_000, replace=False)
sample = feat.iloc[idx][num_cols].astype("float32")
target = pd.Series(labels["is_abandoned"].to_numpy()[idx], index=sample.index)

corr = sample.corr().abs()
mask = np.triu(np.ones(corr.shape, dtype=bool), k=1)
pairs = corr.where(mask).stack().sort_values(ascending=False)
print("\nFeature pairs with correlation above 0.8:")
print(pairs[pairs > 0.8].round(3))

print("\nCorrelation with is_abandoned (sorted by strength):")
tc = sample.corrwith(target)
print(tc.reindex(tc.abs().sort_values(ascending=False).index).round(3))

print("\nMost skewed features (log transform candidates):")
print(sample.skew().sort_values(ascending=False).head(10).round(1))

Cart events per session (percentiles):
count    529865.000000
mean          6.457745
std          11.438531
min           1.000000
50%           3.000000
90%          15.000000
99%          47.000000
99.9%       120.000000
max        1279.000000
dtype: float64

Sessions with more than 100 cart events: 821
Share of all cart events in those sessions (%): 3.96
Abandoned share, heavy vs normal sessions (%):
session_id
False    73.5
True     87.5
Name: is_abandoned, dtype: float64

Users with more than 500 cart events: 118
Share of all cart events from those users (%): 2.77
Abandoned share, heavy vs normal users (%):
user_id
False    73.6
True     87.4
Name: is_abandoned, dtype: float64

Feature pairs with correlation above 0.8:
previous_purchase_count   user_prior_purchases           1.000
cat_prior_carts           cat_prior_purchases            0.990
userprod_prior_carts      userprod_prior_view_to_cart    0.979
cat_prior_carts           cat_prior_removes              0.966
prod_prior_rem

In [10]:
# Row alignment check: features were built in the same order as labels
assert (feat["row_id"].to_numpy() == labels["row_id"].to_numpy()).all()

# Filter columns (NOT model features: they use whole-session/whole-period counts)
heavy_session = labels.groupby("session_id")["row_id"].transform("size") > 100
heavy_user = labels.groupby("user_id")["row_id"].transform("size") > 500

meta = labels[[
    "row_id", "user_id", "session_id", "category_id", "event_time",
    "outcome", "is_abandoned", "censored", "quick_remove", "removed_then_rebought",
]].reset_index(drop=True)
meta["heavy_session"] = heavy_session.to_numpy()
meta["heavy_user"] = heavy_user.to_numpy()

feature_cols = [c for c in feat.columns if c != "row_id"]
model_df = pd.concat([meta, feat[feature_cols].reset_index(drop=True)], axis=1)

# Checks
print("Shape:", model_df.shape)
print("Number of features:", len(feature_cols))
print("Nulls:", int(model_df.isnull().sum().sum()))
print("Feature columns:", feature_cols)

usable = ~model_df["censored"] & ~model_df["heavy_session"] & ~model_df["heavy_user"]
print("\nRows usable for training (not censored, not heavy):", int(usable.sum()))
print("Abandoned share in usable rows (%):",
      round(model_df.loc[usable, "is_abandoned"].mean() * 100, 2))
print("\nUsable rows by month:")
print(model_df.loc[usable, "event_time"].dt.strftime("%Y-%m").value_counts().sort_index())

# Save one CSV
model_df.to_csv(
    "../data/final_model_ready_cart_events_v2.csv",
    index=False,
    date_format="%Y-%m-%d %H:%M:%S",
)
print("\nSaved: ../data/final_model_ready_cart_events_v2.csv")

Shape: (3421733, 72)
Number of features: 60
Nulls: 0
Feature columns: ['views_before_cart', 'carts_before', 'removes_before', 'purchases_before_in_session', 'session_age_min', 'is_first_cart_in_session', 'no_view_before_cart', 'cart_hour', 'cart_day_of_week', 'cart_is_weekend', 'net_cart_value', 'price', 'category_avg_price', 'price_vs_category_avg', 'price_ratio_to_category', 'has_prior_view_of_product', 'price_change_since_view', 'hours_since_last_view', 'previous_purchase_count', 'has_previous_purchase', 'previous_cart_events', 'previous_sessions', 'days_since_first_event', 'hours_since_prev_session', 'category_diversity_before_cart', 'brand_switch_count_before_cart', 'cart_brand_is_real', 'prod_prior_views', 'prod_prior_carts', 'prod_prior_removes', 'prod_prior_purchases', 'prod_prior_buy_rate', 'prod_prior_remove_rate', 'prod_prior_view_to_cart', 'cat_prior_views', 'cat_prior_carts', 'cat_prior_removes', 'cat_prior_purchases', 'cat_prior_buy_rate', 'cat_prior_remove_rate', 'cat_pr